# [실습 3] Evaluate 라이브러리로 평가 파이프라인 만들기


## 실습 개요

> "모델 평가를 accuracy 하나로 끝내지 않고 여러 관점에서 볼 수 있을까?"

이번 실습에서는 `evaluate` 라이브러리로 분류 지표를 계산하고, 생성 결과를 간단한 토큰 겹침 지표로 비교합니다.  
강의에서 다룬 복수 메트릭과 평가 파이프라인 설계 흐름을 코드로 구성합니다.


## 실습 목표

1. `evaluate.load()`로 표준 메트릭을 불러온다.
2. accuracy, precision, recall, f1을 계산한다.
3. 여러 메트릭을 하나의 함수로 묶는다.
4. 생성 결과 평가에서 자동 지표의 한계를 이해한다.
5. [TODO] 복수 메트릭 평가 함수를 완성한다.


## 데이터 출처

- 데이터: 실습용 예측/정답 리스트 직접 구성
- 라이브러리: Hugging Face `evaluate`


## 목차

1. 라이브러리 임포트
2. 단일 메트릭 계산
3. 복수 메트릭 계산
4. 생성 결과 간단 비교
5. [TODO] 평가 함수 완성


---
## 1. 라이브러리 임포트


### 예측값과 정답값 준비

평가 메트릭은 모델 예측값과 기준 정답값을 비교해 계산됩니다. 여기서는 모델이 이미 예측한 결과처럼 `predictions`를 만들고, 실제 정답을 `references`로 둡니다.

평가 파이프라인을 설계할 때는 두 리스트의 길이와 라벨 ID 체계가 서로 일치하는지 먼저 확인해야 합니다. 이 기본 조건이 맞지 않으면 메트릭 값이 의미를 갖기 어렵습니다.

In [5]:
%pip install evaluate
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 11.6 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import warnings
warnings.filterwarnings('ignore')

import evaluate
import pandas as pd

references = [1, 0, 1, 1, 0, 1, 0, 0, 1]
predictions = [1, 0, 1, 0, 0, 1, 1, 0, 0]
print('샘플 수:', len(references))

샘플 수: 9


---
## 2. 단일 메트릭 계산


### accuracy로 기본 성능 확인하기

`evaluate.load('accuracy')`는 Hugging Face Hub에 등록된 표준 accuracy 구현을 불러옵니다. 같은 인터페이스로 여러 메트릭을 불러올 수 있어서 실험 코드가 단순해집니다.

accuracy는 전체 샘플 중 맞춘 비율이라 직관적입니다. 다만 클래스 불균형이 있는 문제에서는 높은 accuracy가 좋은 모델을 의미하지 않을 수 있어 다른 지표와 함께 해석해야 합니다.

In [5]:
accuracy = evaluate.load('accuracy')
accuracy_result = accuracy.compute(predictions=predictions, references=references)
accuracy_result

{'accuracy': 0.6666666666666666}

---
## 3. 복수 메트릭 계산


### 여러 메트릭을 같은 방식으로 계산하기

accuracy, precision, recall, f1을 반복문으로 계산하면 평가 결과를 일관된 형태로 모을 수 있습니다. 실제 실험에서는 단일 점수만 저장하기보다 여러 지표를 함께 기록해야 모델의 강점과 약점을 더 잘 파악할 수 있습니다.

recall은 놓치면 안 되는 양성 샘플을 얼마나 잘 잡는지, precision은 양성이라고 예측한 것 중 얼마나 정확한지를 보여줍니다. f1은 두 값을 균형 있게 요약하는 지표입니다.

여기서는 `average='macro'`를 사용해 클래스별 점수를 단순 평균합니다. 클래스별 샘플 수가 크게 다를 때는 `weighted` 평균이나 클래스별 상세 지표를 함께 확인하는 편이 더 적절할 수 있습니다.

In [11]:
metric_names = ['accuracy', 'precision', 'recall', 'f1']
metric_results = {}

for name in metric_names:
    metric = evaluate.load(name)
    if name == 'accuracy':
        metric_results[name] = metric.compute(
            predictions=predictions, 
            references=references
            )[name]
    else:
        metric_results[name] = metric.compute(
            predictions=predictions,
            references=references,
            average='macro',
        )[name]

pd.DataFrame([metric_results])

,accuracy,precision,recall,f1
0,0.666667,0.675,0.675,0.666667


---
## 4. 생성 결과 간단 비교


### 생성 결과를 단순 지표로 비교해 보기

생성 모델은 정답 문장이 하나로 고정되지 않기 때문에 평가가 더 어렵습니다. 같은 의미를 가진 답변도 표현 방식이 달라질 수 있고, 반대로 단어가 많이 겹쳐도 사실이 틀릴 수 있습니다.

여기서는 예측 문장과 기준 문장의 토큰 겹침 비율을 직접 계산해 생성 평가가 어떤 방식으로 출발하는지 확인합니다. ROUGE, BLEU, BERTScore 같은 지표도 생성 결과와 기준 답변을 어떤 관점에서 비교할지 정의한다는 점에서 같은 문제의식에서 출발합니다.

자동 지표는 빠르게 여러 결과를 비교할 때 유용하지만 사실성, 유용성, 문맥 적합성을 완전히 보장하지는 않습니다. 생성형 AI 결과를 평가할 때는 자동 지표와 사람의 검토 기준을 함께 설계하는 경우가 많습니다.

In [12]:
generated = [
    'pipeline api simplifies model inference',
    'trainer api manages training loop',
]
reference_texts = [
    'pipeline api makes inference simple',
    'trainer manages the model training loop',
]

def token_overlap(pred, ref):
    pred_tokens = set(pred.lower().split())
    ref_tokens = set(ref.lower().split())
    if not ref_tokens:
        return 0.0
    return len(pred_tokens & ref_tokens) / len(ref_tokens)

overlap_scores = [token_overlap(p, r) for p, r in zip(generated, reference_texts)]
print(overlap_scores)

[0.6, 0.6666666666666666]


---
### [TODO] 복수 메트릭 평가 함수 완성

`compute_classification_metrics()` 함수를 완성하세요.

구현 조건:
- 함수 인자는 `predictions`, `references`입니다.
- accuracy, precision, recall, f1을 계산합니다.
- precision/recall/f1에는 `average='binary'`를 사용합니다.
- 결과는 딕셔너리로 반환합니다.


### 반복 평가를 위한 함수 만들기

여러 실험을 반복하다 보면 같은 메트릭 계산 코드를 계속 작성하게 됩니다. 평가 로직을 함수로 묶어 두면 모델이나 데이터가 바뀌어도 같은 기준으로 결과를 비교할 수 있습니다.

반환값을 딕셔너리로 만들면 DataFrame, 로그 파일, 실험 추적 도구에 기록하기 쉽습니다. Trainer의 `compute_metrics` 콜백도 이와 비슷하게 예측값과 정답값을 받아 지표 딕셔너리를 반환하는 구조입니다.

함수 안에서 사용할 평균 방식, 라벨 목록, zero division 처리 같은 세부 설정을 명시해 두면 실험 간 비교가 더 안정적입니다. 평가 코드는 짧아 보여도 실험 결론에 직접 영향을 주기 때문에 재사용 가능한 형태로 관리하는 것이 좋습니다.

In [ ]:
def compute_classification_metrics(predictions, references):
    results = {}
    results['accuracy'] = None
    results['precision'] = None
    results['recall'] = None
    results['f1'] = None
    return results

final_metrics = compute_classification_metrics(predictions, references)
final_metrics

### 평가 결과를 표로 정리하기

메트릭 딕셔너리를 DataFrame으로 바꾸면 여러 실험 결과를 한눈에 비교하기 좋습니다. 모델 이름, 데이터 버전, 하이퍼파라미터 같은 정보와 함께 저장하면 실험 관리가 쉬워집니다.

지금은 결과가 한 줄뿐이지만, 같은 형식으로 여러 모델의 결과를 쌓으면 간단한 리더보드처럼 사용할 수 있습니다.

In [ ]:
print(pd.DataFrame([final_metrics]))